# Figure 3 — formal Figure 2 JSON handoff

Use the pneuma_seeker kernel. The offline sections test a reference planner. The final live section calls the configured model from local .env and executes model-generated SQL on fictional data. No API key is stored in this notebook.


In [ ]:
from pathlib import Path
import sys, json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/intent_spec_example_mimic3_figure3.json").exists())
sys.path.insert(0, str(root / "experiments/figure3"))
from figure3 import load_spec, synthetic_database, run_figure3
spec = load_spec()
print(spec["question"])

## Plan and execute

The cohort has admissions 101–104, with one death (101). Duplicate diagnoses must not increase the count. Multiple admissions per patient count separately. Expected result: 25%.

In [ ]:
db = synthetic_database(spec)
try:
    outcome = run_figure3(spec, db)
    assert outcome["status"] == "success", outcome
    assert outcome["execution_result"]["answer"] == 25.0, outcome
    print(outcome["query_plan"]["sql"])
    print(json.dumps(outcome["execution_result"], indent=2))
finally:
    db.close()

## Regression checks

Exercise empty cohorts, no/all deaths, repeated diagnoses, missing mappings, changed intent, physical schema mismatch, SQL repair limits, and rejection of non-SELECT statements.

In [ ]:
import unittest
from test_figure3 import Figure3Tests
result = unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(Figure3Tests))
assert result.wasSuccessful()

## Live LLM planning and execution

Uses gpt-4.1-mini-2025-04-14 by default (override FIGURE3_LLM_MODEL in local .env). Run All makes paid API requests: one plan, at most one execution repair, and a separate injected-typo repair test. Six checks validate generated SQL on fictional data. Expected answers, reference SQL and fixture rows are not sent to the model. Set RUN_LIVE_LLM=False for offline runs.


In [ ]:
RUN_LIVE_LLM = False
if RUN_LIVE_LLM:
    from llm_planner import run_live_demo
    try:
        live_outcome = run_live_demo()
    except Exception as exc:
        raise RuntimeError("Live setup/provider failed: " + type(exc).__name__) from None
    print(json.dumps(live_outcome, indent=2))
    assert live_outcome["validation_passed"], "Live run failed; inspect status and checks above."
    (root / "experiments/figure3/live_run_result.json").write_text(json.dumps(live_outcome, indent=2) + "\n")
else:
    print("Live API call skipped.")


## Real local MIMIC-III database

The 21 source tables were loaded using Faith’s preparation code. This section makes an LLM call using only the formal IntentSpec and executes SQL on the real database read-only. The complete Figure 4 handoff is saved under git-ignored local_results/real_run.json. Patient rows and real answers are not sent to the LLM. The independent reference check is specific to this delivered HF question; it is not the Figure 4 reviewer.


In [ ]:
RUN_REAL_DATABASE = False
if RUN_REAL_DATABASE:
    from run_real import run_real, DEFAULT_OUTPUT
    real_outcome = run_real()
    print("Status:", real_outcome["status"])
    print("Reference matches:", real_outcome["reference_check"]["matches"])
    print("Full result saved locally:", DEFAULT_OUTPUT)
    assert real_outcome["reference_check"]["matches"], "Inspect local result before handoff"


## Integrated Figure 2 → Figure 3 → Figure 4 handoff

Calls the teammate’s actual IntentPreparer, then uses its newly generated IntentSpec. No fixed JSON supplies planning requirements here; only its question is used as the default text. Clarification/unanswerable outcomes stop execution. Figure 3 can send at most one AdjustmentRequest back to Figure 2. SQL repair has a separate one-repair limit. Results stay under git-ignored local_results/. Model output may select different definitions than the fixed example; inspect the full IntentSpec before claiming answer correctness. Current Figure 3 execution supports single numeric results.


In [ ]:
RUN_INTEGRATED_WORKFLOW = True
if RUN_INTEGRATED_WORKFLOW:
    from workflow import run_integrated
    question = spec["question"]
    integrated_outcome = run_integrated(question)
    print("Workflow status:", integrated_outcome["status"])
    print("Upstream adjustments:", integrated_outcome["upstream_adjustments_used"])
    print("Full local handoff: experiments/figure3/local_results/figure4_integrated_handoff.json")
